In [7]:
from aviary.core import Environment, Message, Tool, ToolRequestMessage, ToolSelector, ToolResponseMessage
from ldp.agent import Agent
from ldp.graph import LLMCallOp
from ldp.alg import RolloutManager
import os
from helper.problem import ProblemState, examples
from helper.units import UNIT_ALIASES
from helper.tools import *
from helper.schemas import *

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [9]:
TOOLS  = {
#   "convert_units":    convert_units,
  "molar_mass":         lookup_molar_mass,
  "compute_moles":      compute_moles,
  "mass_from_moles":    mass_from_moles,
  "mix_concentration":  mix_concentration,
  "dilution": dilution,
}

class ScienceEnv(Environment[ProblemState]):
    def __init__(self, problem: ProblemState):
          super().__init__()
          self.problem = problem
          self._oracle = None
        
    async def reset(self):
        self.tools = [Tool.from_function(fun) 
                      for _, fun in TOOLS.items()]
        prompt = self.problem.prompt or build_prompt(self.problem)
        print(f"[RESET] prompt: {prompt}")
        return [Message(content=prompt)], self.tools

    async def step(self, action):
        if isinstance(action, ToolRequestMessage) and action.tool_calls:
            # run the tool
            raw = await self.exec_tool_calls(action)

            # extract the text
            if isinstance(raw, ToolResponseMessage):
                text = raw.content
            elif isinstance(raw, list) and all(isinstance(r, ToolResponseMessage) for r in raw):
                text = " ".join(r.content for r in raw)
            else:
                text = str(raw)

            # wrap it as a normal assistant Message
            msg = Message(content=text)

            # compute reward immediately
            agent   = text.strip()
            expected = self.problem.expected.strip()
            reward   = float(agent == expected)

            # DONE in one step
            return [msg], reward, True, False


In [10]:
SYSTEM = Message(
    role="system",
    content=(
        "You are a scientific assistant. Plan and execute multiple steps:\n"
        "1) Decide which function to call next.\n"
        "2) Call it with valid JSON args.\n"
        "3) Observe the result.\n"
        "4) Repeat until you have a final answer, then reply with that answer."
    ),
)

In [11]:
# --- Step 2: Simple Agent ---
class AgentState:
    """A container for maintaining agent state across interactions."""

    def __init__(self, messages, tools):
        self.messages = messages
        self.tools = tools


class SimpleAgent(Agent):
    def __init__(self):
        self.tool_selector = ToolSelector(model_name="gpt-4o", accum_messages=False)

    async def init_state(self, tools):
        return AgentState(messages=[SYSTEM], tools=tools)

    async def get_asv(self, agent_state, obs):
        """Take an action, observe new state, return value"""
        history = agent_state.messages + obs
        action = await self.tool_selector(history, agent_state.tools)
        new_state = AgentState(messages=history + [action], tools=agent_state.tools)
        return action, new_state, 0.0

In [12]:
agent = SimpleAgent()

for problem in examples:
    env = ScienceEnv(problem)

    obs, tools    = await env.reset()
    agent_state   = await agent.init_state(tools)

    action, agent_state, _ = await agent.get_asv(agent_state, obs)
    obs, reward, done, _   = await env.step(action)
    print("Final answer by agent:", obs[-1].content)
    print("Expected answer:", problem.expected)
    print("Reward:", reward, "\n")


[DEBUG build_prompt] task='molar_mass', params={'formula': 'H2O'}
[RESET] prompt: What is the molar mass of H2O?
Final answer by agent: 18.015 g/mol
Expected answer: 18.015 g/mol
Reward: 1.0 

[DEBUG build_prompt] task='compute_moles', params={'mass': 18.015, 'formula': 'H2O'}
[RESET] prompt: Given 18.015 g of H2O (stoich ratio 1:1), how many moles of product are formed?
Final answer by agent: 1.0000 mol
Expected answer: 1.0000 mol
Reward: 1.0 

[DEBUG build_prompt] task='mass_from_moles', params={'moles': 0.5, 'molar_mass': 44.01}
[RESET] prompt: Convert 0.5 mol to grams using a molar mass of 44.01 g/mol.
Final answer by agent: 22.005 g
Expected answer: 22.005 g
Reward: 1.0 

[DEBUG build_prompt] task='mix_concentration', params={'C1': 0.1, 'V1': 0.05, 'C2': 0.2, 'V2': 0.1}
[RESET] prompt: Mix 0.05 L of 0.1 M solution with 0.1 L of 0.2 M solution; what is the final concentration?
Final answer by agent: 0.167 M
Expected answer: 0.167 M
Reward: 1.0 

[DEBUG build_prompt] task='dilution'